# Tempo Run 2025 — Train QLoRA

Config: `configs/qlora_qwen3_0_6b.yaml`
- 4-bit NF4 + double quant + LoRA r=32 α=16
- Effective batch 32, 3 epochs
- max_seq 2048
- A100 ~30-60 phút cho 4491 mẫu

**Lưu ý**: checkpoint sẽ được lưu vào `artifacts/qlora_qwen3_0_6b/`. Cell cuối copy sang Drive (nếu có mount).

In [ ]:
%cd /content/finetune_1B_MCQ_VN
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")
import torch
assert torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory >= 15e9, \
    "Cần GPU ≥15GB VRAM"

In [ ]:
# 1. Xem config trước khi train
!cat configs/qlora_qwen3_0_6b.yaml

In [ ]:
# 2. (Optional) Smoke test với max_steps nhỏ — train 50 step để check pipeline OK trước
# Bỏ comment nếu muốn:
# !python -c "
# import yaml
# from temprun.utils import load_config
# cfg = load_config('configs/qlora_qwen3_0_6b.yaml')
# cfg['training']['max_steps'] = 50
# cfg['run_name'] = 'qlora_smoke'
# from temprun.train import train
# train(cfg)
# "

In [ ]:
# 3. Train đầy đủ — QLoRA
!python scripts/train.py --config configs/qlora_qwen3_0_6b.yaml

In [ ]:
# 4. Copy sang Drive để persist
import os, shutil
DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/temprun_runs")
if os.path.isdir("/content/drive/MyDrive"):
    src = "artifacts/qlora_qwen3_0_6b"
    dst = os.path.join(DRIVE_ROOT, "qlora_qwen3_0_6b")
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"Copied to: {dst}")
else:
    print("Drive not mounted; skipping copy.")